# ReAct Agent — Live Coding Demo
### AI Engineering · Lecture 4 · Nebius Academy

One query. Two approaches. See the difference.

**The question we'll answer throughout:**
> *"What is the current #1 model on the LMSYS Chatbot Arena leaderboard, and who made it?"*

This question has a definite correct answer that changes over time — the LLM's training data is stale. It cannot answer reliably without searching.


## 1 · Setup

In [1]:
%pip install openai --quiet


In [ ]:
import os, re, json, textwrap, urllib.request, urllib.parse
from openai import OpenAI

# ── API key
# Colab: Secrets panel (key icon) → add NEBIUS_API_KEY
# Local: export NEBIUS_API_KEY="your-key-here"
# ── Nebius API config ─────────────────────────────────────────────────────
NEBIUS_API_KEY =    # ← paste your token
NEBIUS_BASE_URL = "https://api.studio.nebius.ai/v1/"

client = OpenAI(
    base_url=NEBIUS_BASE_URL,
    api_key=NEBIUS_API_KEY,
)


MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
QUESTION = "What is the current #1 model on the LMSYS Chatbot Arena leaderboard, and who made it?"

print(f"✓ Ready  |  model: {MODEL}")
print(f"  Question: {QUESTION}")


✓ Ready  |  model: meta-llama/Meta-Llama-3.1-8B-Instruct
  Question: What is the current #1 model on the LMSYS Chatbot Arena leaderboard, and who made it?


---
## 2 · Attempt 1: bare LLM

No tools. Just ask the model directly.


In [4]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": QUESTION}],
    max_tokens=200,
    temperature=0,
)

print("Answer (no tools):")
print(response.choices[0].message.content.strip())


Answer (no tools):
I cannot verify the current #1 model on the LMSYS Chatbot Arena leaderboard. Is there anything else I can help you with?


The model will give a stale answer, hedge, or guess. The Chatbot Arena leaderboard updates continuously — training data can't keep up.

This is the gap we're about to close.


---
## 3 · The tool

A tool is a Python function + a description the LLM reads.

We need one tool: web search. We'll use the Wikipedia API — no key required, works in Colab.


In [5]:
def wikipedia_search(query: str) -> str:
    """Search Wikipedia and return the article summary."""
    try:
        # Find best matching article
        search_url = (
            "https://en.wikipedia.org/w/api.php"
            f"?action=query&list=search&srsearch={urllib.parse.quote(query)}"
            "&srlimit=1&format=json"
        )
        req = urllib.request.Request(search_url, headers={"User-Agent": "ReActDemo/1.0"})
        with urllib.request.urlopen(req, timeout=8) as r:
            results = json.loads(r.read())["query"]["search"]

        if not results:
            return f"No Wikipedia article found for: '{query}'"

        title = results[0]["title"]

        # Fetch summary
        summary_url = (
            "https://en.wikipedia.org/api/rest_v1/page/summary/"
            + urllib.parse.quote(title.replace(" ", "_"))
        )
        req2 = urllib.request.Request(summary_url, headers={"User-Agent": "ReActDemo/1.0"})
        with urllib.request.urlopen(req2, timeout=8) as r2:
            data = json.loads(r2.read())

        extract = data.get("extract", "No extract available.")
        return f"[Wikipedia: {title}]\n{extract[:600]}"

    except Exception as e:
        return f"Search error: {e}"


# Smoke test
print(wikipedia_search("LMSYS Chatbot Arena leaderboard"))


[Wikipedia: LMArena]
Arena is a public, web-based platform that evaluates large language models (LLMs). Users enter prompts for two anonymous models to respond to and vote on the model that gave the better response, after which the models' identities are revealed. Users can also choose models to test themselves.


---
## 4 · The ReAct prompt

This is the entire "framework" — a text format the model must follow:

```
Thought:      what do I know, what do I need next
Action:       tool_name(argument)
Observation:  <filled in by our code — the actual tool result>
... repeat ...
Final Answer: <answer built from Observations only>
```

Two things make this work:
- The prompt instructs the model to stop after `Action:` — our code runs the tool and provides the `Observation:`
- `stop=["Observation:"]` in the API call enforces this — the model cannot write its own tool result


In [6]:
SYSTEM_PROMPT = """You are a ReAct agent. Answer questions by alternating Thought and Action steps.

You have one tool:
  wikipedia_search(query: str) — searches Wikipedia, returns an article summary

Use this format exactly:

Thought: <what you know and what you need to find out>
Action: wikipedia_search(<your search query>)

The system will then provide:
Observation: <tool result>

Repeat until you have enough information, then write:

Thought: I have enough information to answer.
Final Answer: <your answer, based only on what you observed>

Important: never write Observation: yourself. Never answer from memory.
"""


---
## 5 · The loop

Three functions — this is the entire agent:


In [8]:
def call_llm(messages):
    """One LLM call. Stops before Observation: so the model can't invent the tool result."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=300,
        temperature=0,
        stop=["Observation:"],   # critical: hand control back to us
    )
    return response.choices[0].message.content.strip()


def parse_action(text):
    """Extract tool_name and argument from 'Action: tool_name(argument)'."""
    match = re.search(r"Action:\s*wikipedia_search\((.+?)\)", text, re.DOTALL)
    if not match:
        return None
    return match.group(1).strip().strip("'")


def run_agent(question, max_steps=6):
    print(f"Question: {question}\n{'─'*60}")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]

    for step in range(1, max_steps + 1):
        print(f"\n[Step {step}]")

        output = call_llm(messages)
        print(output)

        if "Final Answer:" in output:
            answer = output.split("Final Answer:")[-1].strip()
            print(f"\n{'─'*60}\n✓ {answer}")
            return answer

        argument = parse_action(output)
        if argument is None:
            print("[!] No action found — stopping.")
            break

        observation = wikipedia_search(argument)
        print(f"Observation: {observation[:300]}...")

        messages.append({
            "role": "assistant",
            "content": output + f"\nObservation: {observation}",
        })

    return "Max steps reached."


---
## 6 · Attempt 2: ReAct agent

Same question. Now with the loop.


In [9]:
run_agent(QUESTION)


Question: What is the current #1 model on the LMSYS Chatbot Arena leaderboard, and who made it?
────────────────────────────────────────────────────────────

[Step 1]
Thought: I need to find out the current #1 model on the LMSYS Chatbot Arena leaderboard and its creator. I'm not aware of the current leaderboard rankings or the models that are participating.
Action: wikipedia_search("LMSYS Chatbot Arena leaderboard")
Observation: No Wikipedia article found for: '"LMSYS Chatbot Arena leaderboard"'...

[Step 2]
Thought: Since there's no Wikipedia article, I'll try searching for the LMSYS Chatbot Arena itself to see if I can find any information about the leaderboard or the top models.
Action: wikipedia_search("LMSYS Chatbot Arena")
Observation: No Wikipedia article found for: '"LMSYS Chatbot Arena"'...

[Step 3]
Thought: I'll try searching for the LMSYS Chatbot Arena on a different platform, such as the official website or a forum related to the arena.
Action: wikipedia_search("LMSYS Chat

'The current #1 model on the LMArena leaderboard is Gemini, developed by Google DeepMind.'

---

The model searched, read the result, and answered from the Observation — not from memory.

That's ReAct: **Re**asoning + **Act**ing. Every production agent framework (LangGraph, smolagents, CrewAI) is scaffolding around this same loop.
